# Mapeamento Eleitoral e Visualização Geospacial com Folium


In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import pathlib

In [2]:
data_path = pathlib.Path('./data')
sjdr_votacao_csv_path = data_path / 'votacao_sjdr.csv' 

In [3]:
df = pd.read_csv(sjdr_votacao_csv_path, sep=',')
df.head()

,DT_GERACAO_x,HH_GERACAO_x,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_PLEITO,DT_PLEITO,NR_TURNO_x,CD_ELEICAO,DS_ELEICAO_x,...,DS_SITU_LOCALIDADE,CD_SITU_SECAO_ACESSIBILIDADE,DS_SITU_SECAO_ACESSIBILIDADE,QT_ELEITOR_SECAO,QT_ELEITOR_ELEICAO_FEDERAL,QT_ELEITOR_ELEICAO_ESTADUAL,QT_ELEITOR_ELEICAO_MUNICIPAL,NR_LOCAL_VOTACAO_ORIGINAL,NM_LOCAL_VOTACAO_ORIGINAL,DS_ENDERECO_LOCVT_ORIGINAL
0,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,238,414,0,0,1023,COLEGIO NOSSA SENHORA DAS DORES,"RUA COMENDADOR BASTOS, 84"
1,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,238,415,415,0,1023,COLEGIO NOSSA SENHORA DAS DORES,"RUA COMENDADOR BASTOS, 84"
2,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,323,325,325,0,1716,SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI,"PRAÇA BOM JESUS DE MATOZINHOS, S/N"
3,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,323,325,0,0,1716,SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI,"PRAÇA BOM JESUS DE MATOZINHOS, S/N"
4,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,359,356,356,0,1864,ESCOLA ESTADUAL GOVERNADOR MILTON CAMPOS,"AV. SETE DE SETEMBRO, S/N"


In [4]:
df_turno1 = df[df["NR_TURNO_x"] == 1]
df_turno2 = df[df["NR_TURNO_x"] == 2]

count_turno1 = len(df_turno1)
count_turno2 = len(df_turno2)

print("Qtd de registros turno 1:", count_turno1)
print("Qtd de registros turno 2:", count_turno2)

Qtd de registros turno 1: 3702
Qtd de registros turno 2: 1710


**Conta a quantidade de zonas eleitorais**

SJDR tem somente uma zona eleitoral (zona 328)

In [5]:
df_zona_1T = df_turno1.groupby('NR_ZONA')
df_zona_2T = df_turno2.groupby('NR_ZONA')

print(f'Zonas 1T: {len(df_zona_1T)}') 
print(f'Zonas 2T: {len(df_zona_2T)}')

Zonas 1T: 1
Zonas 2T: 1


**Conta a quantidade de secoes eleitorais**

A zona 328 tem 241 secoes eleitorais, contudo ela abrange cidades da regiao de SJDR, nao somente SJDR. Como filtramos apenas por SJDJ, e normal que tenham menos secoes que o total de secoes da zona.

In [40]:
secoes_1T = df_zona_1T['NR_SECAO'].nunique() 
secoes_2T = df_zona_2T['NR_SECAO'].nunique() 

print(f"Qtd secoes 1T: {secoes_1T.iloc[0]}")
print(f"Qtd secoes 2T: {secoes_2T.iloc[0]}")

Qtd secoes 1T: 218
Qtd secoes 2T: 218


## Limpeza, Filtragem e Unificação dos Dados

### Agrupa o Total de Votos por Candidato em Cada Local de Votação em Ambos os Turnos

In [57]:
df_locais_1T = (
    df_turno1.groupby(['NR_LOCAL_VOTACAO_x', 'NM_VOTAVEL', "NR_VOTAVEL"], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by=['NR_LOCAL_VOTACAO_x', 'QT_VOTOS'], ascending=[True, False],)
)

df_locais_1T.head(10)

,NR_LOCAL_VOTACAO_x,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS
2,1023,FLAVIO BOLSONARO,22,3412
3,1023,LULA,13,2376
1,1023,ESCRITOR AUGUSTO CURY,70,160
5,1023,RENAN SANTOS,14,160
6,1023,RONALDO CAIADO,55,118
4,1023,Nulo,96,114
9,1023,ZEMA,30,70
0,1023,Branco,95,66
7,1023,SAMARA,80,4
8,1023,VETERINÁRIO WILSON GRASSI,35,2


**Calcula o percentual de votos de cada candidato dentro da seção**

In [58]:
total_votos_local2T = df_locais_1T.groupby('NR_LOCAL_VOTACAO_x')['QT_VOTOS'].transform('sum')
df_locais_1T['PCT_VOTOS_LOCAL'] = ((df_locais_1T['QT_VOTOS'] / total_votos_local2T) * 100.00).round(2)

df_locais_1T.head(10)

,NR_LOCAL_VOTACAO_x,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,PCT_VOTOS_LOCAL
2,1023,FLAVIO BOLSONARO,22,3412,52.64
3,1023,LULA,13,2376,36.66
1,1023,ESCRITOR AUGUSTO CURY,70,160,2.47
5,1023,RENAN SANTOS,14,160,2.47
6,1023,RONALDO CAIADO,55,118,1.82
4,1023,Nulo,96,114,1.76
9,1023,ZEMA,30,70,1.08
0,1023,Branco,95,66,1.02
7,1023,SAMARA,80,4,0.06
8,1023,VETERINÁRIO WILSON GRASSI,35,2,0.03


In [59]:
df_locais_2T = (
    df_turno2.groupby(['NR_LOCAL_VOTACAO_x', 'NM_VOTAVEL', 'NR_VOTAVEL'], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by=['NR_LOCAL_VOTACAO_x', 'QT_VOTOS'], ascending=[True, False],)
)

df_locais_2T.head(10)

,NR_LOCAL_VOTACAO_x,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS
1,1023,FLAVIO BOLSONARO,22,3704
2,1023,LULA,13,2872
3,1023,Nulo,96,104
0,1023,Branco,95,54
5,1058,FLAVIO BOLSONARO,22,1124
6,1058,LULA,13,792
7,1058,Nulo,96,42
4,1058,Branco,95,20
9,1066,FLAVIO BOLSONARO,22,232
10,1066,LULA,13,160


**Calcula o percentual de votos de cada candidato dentro da seção**

In [60]:
total_votos_local2T = df_locais_2T.groupby('NR_LOCAL_VOTACAO_x')['QT_VOTOS'].transform('sum')
df_locais_2T['PCT_VOTOS_LOCAL'] = ((df_locais_2T['QT_VOTOS'] / total_votos_local2T) * 100.00).round(2)

df_locais_2T.head(10)

,NR_LOCAL_VOTACAO_x,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,PCT_VOTOS_LOCAL
1,1023,FLAVIO BOLSONARO,22,3704,55.00
2,1023,LULA,13,2872,42.65
3,1023,Nulo,96,104,1.54
0,1023,Branco,95,54,0.80
5,1058,FLAVIO BOLSONARO,22,1124,56.83
6,1058,LULA,13,792,40.04
7,1058,Nulo,96,42,2.12
4,1058,Branco,95,20,1.01
9,1066,FLAVIO BOLSONARO,22,232,57.43
10,1066,LULA,13,160,39.60


## Choropleth Maps do 1º Turno

**Transforma a latitude e longitude das zonas eleitorais em float**

In [61]:
df_turno1['NR_LATITUDE'] = pd.to_numeric(
    df_turno1['NR_LATITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
)
df_turno2['NR_LATITUDE'] = pd.to_numeric(
    df_turno1['NR_LATITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
)

df_turno1['NR_LONGITUDE'] = pd.to_numeric(
    df_turno1['NR_LONGITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
)
df_turno2['NR_LONGITUDE'] = pd.to_numeric(
    df_turno1['NR_LONGITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
)

**Obtem o mapa**

In [62]:
map = folium.Map(
    location=[-21.1365, -44.2619], # SJDR
    zoom_start=20,
    tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors' 
)

**Coloca marcadores nas secoes de votacao**

In [63]:
map_cols = [
    'NM_LOCAL_VOTACAO',
    "NR_LOCAL_VOTACAO_x",
    'NR_LATITUDE',
    'NR_LONGITUDE',
]

df_lat_lon_locais = df_turno1[map_cols].drop_duplicates()

for row in df_lat_lon_locais.itertuples(index=False):
    lat = row.NR_LATITUDE
    lon = row.NR_LONGITUDE
    local = row.NM_LOCAL_VOTACAO
    
    folium.Marker(
        location=[lat, lon],
        popup=f'{local} ({row.NR_LOCAL_VOTACAO_x})',
    ).add_to(map)

display(map)

**Atribui paleta de cores para cada candidato**

In [64]:
def candidate_color_func(party:int):
    # Lula
    if party == 13:
        return '#ff0000'
    
    # Renan Santos
    if party == 14:
        return "#f0d01a"
    
    # Hertz Dias
    if party == 16:
        return "#780f07"
   
    # Edmilson Costa
    if party == 21:
        return "#fc4521"
    
    # Flavio Bolsonaro
    if party == 22:
        return "#0c7a0a"
   
    # Clariana Barão 
    if party == 27:
        return "#80d7ff"
    
    # Rui Costa Pimenta  
    if party == 29:
        return "#9c2308"
    
    # Romeu Zema
    if party == 30:
        return "#e66909"
    
    # Veterinário Wilson Grassi
    if party == 35:
        return "#187031"
   
    # Renato Caiado
    if party == 55:
        return "#0306a6"
    
    # Augusto Cury
    if party == 70:
        return "#077375"
    
    # Samara Martins
    if party == 80:
        return "#000000"
    
    # Branco 
    if party == 95:
        return "#ffffff"
    
    # Nulo
    if party == 96:
        return "#a3a3a3"
    
    # Outros? (Nao é pra ter)
    return "#ff00dd"

In [65]:
idx_winners = df_locais_1T.groupby('NR_LOCAL_VOTACAO_x')['PCT_VOTOS_LOCAL'].idxmax()
df_locais_1T_winners = df_locais_1T.loc[idx_winners].reset_index(drop=True)

df_locais_1T_winners.head(10)

,NR_LOCAL_VOTACAO_x,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,PCT_VOTOS_LOCAL
0,1023,FLAVIO BOLSONARO,22,3412,52.64
1,1058,FLAVIO BOLSONARO,22,1014,53.65
2,1066,FLAVIO BOLSONARO,22,196,53.26
3,1074,LULA,13,878,44.25
4,1082,LULA,13,524,45.80
5,1090,FLAVIO BOLSONARO,22,284,45.51
6,1104,LULA,13,1626,49.51
7,1112,LULA,13,2318,44.30
8,1120,FLAVIO BOLSONARO,22,668,45.32
9,1139,FLAVIO BOLSONARO,22,996,44.58


In [69]:
map_ganhadores = folium.Map(
    location=[-21.1365, -44.2619], # SJDR
    zoom_start=20,
    tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors' 
)

for row in df_locais_1T_winners.itertuples(index=False):
    lat = df_lat_lon_locais.loc[df_lat_lon_locais.NR_LOCAL_VOTACAO_x == row.NR_LOCAL_VOTACAO_x]["NR_LATITUDE"].item()
    lon = df_lat_lon_locais.loc[df_lat_lon_locais.NR_LOCAL_VOTACAO_x == row.NR_LOCAL_VOTACAO_x]["NR_LONGITUDE"].item()
    
    pct = row.PCT_VOTOS_LOCAL
    party = row.NR_VOTAVEL
    color = candidate_color_func(party)
    nm_candidato = row.NM_VOTAVEL
    
    folium.Circle(
        location=[lat, lon],
        radius=200 * (pct / 100.0),
        popup=f"{nm_candidato} ({pct}%) ({row.NR_LOCAL_VOTACAO_x})",
        color=color,
        fill=True,
        fill_opacity=0.2,
    ).add_to(map_ganhadores)
    
map_ganhadores.save("mapa_ganhadores.html")